In [1]:
import os
import nglview
import numpy as np
from openff.toolkit import ForceField, Molecule, Topology
from openff.units import unit
from openff.interchange import Interchange
from openff.interchange.components._packmol import UNIT_CUBE, RHOMBIC_DODECAHEDRON, pack_box
from openff.interchange.drivers import get_openmm_energies
from openff.interchange.drivers.all import get_summary_data
from rdkit import Chem

In [2]:
IDs=["1654"]

ID = IDs[0]
print(ID)
#seq_type = "binder"
seq_type = "nonbinder"
#seq_type = "neg_low_pkt"
#seq_type = "neg_fail_gate"
#seq_type = "binder" ["3103", "3104", "3108"]
#seq_type = "nonbinder" # ["0777", "0827", "1824", "1939", "0664"] ["0052", "0091", "0246", "0269", "0272"]
#seq_type = "neg_low_pkt" # ["0008", "0393", "0482", "1331", "1931"] ["0065", "0947", "0948", "1413"]
#seq_type = "neg_fail_gate" # ["0715", "1094", "1682", "1739", "1941"] ["0014", "0263", "0540", "0997", "1745"]

1654


In [3]:
if seq_type == "binder":
    # pdb_file = os.path.abspath(f"binders/seq{ID}_binder/protein_seq{ID}_binder_fixed_H.pdb")
    # sdf_file = f"binders/seq{ID}_binder/ligand_seq{ID}_binder.sdf"
    pdb_file = os.path.abspath(f"binders/pair_{ID}_binder/protein_pair{ID}_fixed_H.pdb")
    sdf_file = f"binders/pair_{ID}_binder/ligand_pair{ID}.sdf"
    folder_name = "binders"
    extension = seq_type
elif seq_type == "nonbinder":
    # pdb_file = os.path.abspath(f"nonbinders/seq{ID}_nb/protein_seq{ID}_nb_fixed_H.pdb")
    # sdf_file = f"nonbinders/seq{ID}_nb/ligand_seq{ID}_nb.sdf"
    pdb_file = os.path.abspath(f"nonbinders/pair_{ID}_nb/protein_pair{ID}_fixed_H.pdb")
    sdf_file = f"nonbinders/pair_{ID}_nb/ligand_pair{ID}.sdf"
    folder_name = "nonbinders"
    extension = "nb"
elif seq_type == "neg_low_pkt":
    pdb_file = os.path.abspath(f"neg_low_pkt/pair_{ID}_low_pkt/protein_pair{ID}_fixed_H.pdb")
    sdf_file = f"neg_low_pkt/pair{ID}_low_pkt/ligand_pair{ID}.sdf"
    folder_name = "neg_low_pkt"
    extension = "low_pkt"
elif seq_type == "neg_fail_gate":
    pdb_file = os.path.abspath(f"neg_fail_gate/pair_{ID}_fail_gate/protein_pair{ID}_fixed_H.pdb")
    sdf_file = f"neg_fail_gate/pair{ID}_fail_gate/ligand_pair{ID}.sdf"
    folder_name = "neg_fail_gate"
    extension = "fail_gate"

## Prep components - protein+ligand

### Ligand

In [4]:
if seq_type == "binder":
    #ligand = Molecule.from_file(f"binders/seq{ID}_binder/ligand_seq{ID}_binder.sdf")
    ligand = Molecule.from_file(f"binders/pair_{ID}_binder/ligand_pair{ID}.sdf")
    name_prefix = "pair"
elif seq_type == "nonbinder":
    #ligand = Molecule.from_file(f"nonbinders/seq{ID}_nb/ligand_seq{ID}_nb.sdf")
    ligand = Molecule.from_file(f"nonbinders/pair_{ID}_nb/ligand_pair{ID}.sdf")
    name_prefix = "pair"
elif seq_type == "neg_low_pkt":
    ligand = Molecule.from_file(f"neg_low_pkt/pair_{ID}_low_pkt/ligand_pair{ID}.sdf")
elif seq_type == "neg_fail_gate":
    ligand = Molecule.from_file(f"neg_fail_gate/pair_{ID}_fail_gate/ligand_pair{ID}.sdf")

In [5]:
ligand.name = "LIG"
ligand.visualize(backend="nglview")

NGLWidget()

In [6]:
ligand_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=[ligand],
)

### Protein

In [7]:
protein_full = Topology.from_pdb(pdb_file)
protein_full.n_molecules

1

In [8]:
protein = protein_full.molecule(0)
protein.name = "protein"
view_protein = protein.visualize(backend="nglview")
view_protein.clear_representations()
view_protein.add_representation("cartoon", selection='protein')
view_protein

NGLWidget()

In [9]:
ff14sb = ForceField("ff14sb_off_impropers_0.0.3.offxml")

In [10]:
protein_intrcg = Interchange.from_smirnoff(
    force_field=ff14sb,
    topology=protein.to_topology(),
)

### Dock ligand + protein

In [11]:
docked_intrcg = protein_intrcg.combine(ligand_intrcg) # dock

# visualize
w = docked_intrcg.visualize()
w.clear_representations()
w.add_representation(
    "cartoon",
    radius=0.1,
    selection=[*range(protein_intrcg.topology.n_atoms)],
)
w.add_representation(
    "licorice",
    selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)],
)
w

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:104: InterchangeCombinationWarning: Interchange object combination is complex and may produce strange results outside of use cases it has been tested in. Use with caution and thoroughly validate results!
  warnings.warn(
/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:84: InterchangeCombinationWarning: Found electrostatics 1-4 scaling factors of 5/6 with slightly different rounding (0.833333 and 0.8333333333). This likely stems from OpenFF using more digits in rounding 1/1.2. The value of 0.8333333333 will be used, which may or may not introduce small errors. 
  warnings.warn(


NGLWidget()

Check partial charge of protein+ligand complex

In [12]:
total_charge = round(sum(docked_intrcg["Electrostatics"].charges.values()), 3)

assert total_charge == protein.total_charge + ligand.total_charge, (
    f"Total charge of the system is {total_charge}, not {protein.total_charge + ligand.total_charge}"
)
total_charge

<Quantity(-5.0, 'elementary_charge')>

### Solvent

In [13]:
total_charge_e = float(total_charge.m)

if total_charge_e < 0:
    counterion = Molecule.from_smiles("[Na+]")
    counterion.name = "NA"
    n_counterions = int(round(abs(total_charge_e)))
elif total_charge_e > 0:
    counterion = Molecule.from_smiles("[Cl-]")
    counterion.name = "CL"
    n_counterions = int(round(abs(total_charge_e)))
else:
    counterion = None
    n_counterions = 0
    
water = Molecule.from_smiles("O")
water.name = "SOL"
water.generate_conformers(n_conformers=1)

In [14]:
#box_shape = "cube"
box_shape = "dodecahedron"

xyz = protein.conformers[0].to(unit.nanometer).m  # (N, 3) in nm, unitless array
centroid = xyz.mean(axis=0)
protein_radius_nm = np.sqrt(((xyz - centroid) ** 2).sum(axis=1).max())

buffer_nm = 2.0 # distance from edge to center

scale_nm = 2.0 * protein_radius_nm + buffer_nm # "box length" of unit cell

if box_shape == "cube":
    
    box_vectors = np.eye(3) * scale_nm * unit.nanometer
    
    # ----------------------------
    # Compute water count for ~1 g/mL
    # ----------------------------
    V_box_nm3 = scale_nm ** 3
    
    # crude displaced volume estimate (protein roughly spherical)
    V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)
    
    waters_per_nm3 = 33.4  # ~1 g/mL at 300 K
    n_water = int(waters_per_nm3 * max(V_box_nm3 - V_solute_nm3, 0.0))
    
    # (Optional) add a little cushion if you find voids / low density
    # n_water = int(1.05 * n_water)
    
    print(f"Box length: {scale_nm:.3f} nm")
    print(f"Packing waters: {n_water}, counterions: {n_counterions}")
    
    # ----------------------------
    # Pack the box
    # ----------------------------
    molecules = [water]
    number_of_copies = [n_water]
    
    if counterion is not None and n_counterions > 0:
        molecules.append(counterion)
        number_of_copies.append(n_counterions)

    packed_topology = pack_box(
        solute=docked_intrcg.topology,
        molecules=molecules,
        number_of_copies=number_of_copies,
        box_vectors=box_vectors,
        center_solute=True,
        tolerance=2.0 * unit.angstrom,
        working_directory=f"/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{name_prefix}_{ID}_{extension}/packmol_solv_box",
        retain_working_files=True,
    )
    
    print("Box length (nm):", scale_nm)
    print("Total molecules packed:", packed_topology.n_molecules)

elif box_shape == "dodecahedron":

    # ----------------------------
    # 2) Rhombic dodecahedron box vectors
    # ----------------------------
    # RHOMBIC_DODECAHEDRON is a 3x3 set of *reduced* vectors; scaling sets the actual box size.
    box_vectors = (scale_nm * RHOMBIC_DODECAHEDRON) * unit.nanometer
    
    # ----------------------------
    # 3) Compute water count for ~1 g/mL using the *true* triclinic volume
    # ----------------------------
    # Convert box vectors to unitless nm matrix for determinant
    box_nm = box_vectors.to(unit.nanometer).m  # shape (3,3)
    V_box_nm3 = abs(np.linalg.det(box_nm))
    
    # crude displaced volume estimate (protein roughly spherical)
    V_solute_nm3 = (4.0 / 3.0) * np.pi * (protein_radius_nm ** 3)
    
    waters_per_nm3 = 33.4  # ~1 g/mL
    n_water = int(waters_per_nm3 * max(V_box_nm3 - V_solute_nm3, 0.0))
    
    print(f"Protein radius: {protein_radius_nm:.3f} nm")
    print(f"Dodecahedron scale: {scale_nm:.3f} nm")
    print(f"Box volume: {V_box_nm3:.2f} nm^3")
    print(f"Packing waters: {n_water}, counterions: {n_counterions}")
    
    # ----------------------------
    # 4) Pack (solute + water + optional counterions)
    # ----------------------------
    molecules = [water]
    number_of_copies = [n_water]
    
    if counterion is not None and n_counterions > 0:
        molecules.append(counterion)
        number_of_copies.append(n_counterions)
    
    packed_topology = pack_box(
        solute=docked_intrcg.topology,
        molecules=molecules,
        number_of_copies=number_of_copies,
        box_vectors=box_vectors,
        center_solute=True,
        tolerance=2.0 * unit.angstrom,
        working_directory=f"/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{name_prefix}_{ID}_{extension}/packmol_solv_dodecahedron",
        retain_working_files=True,
    )
    
    print("Total molecules packed:", packed_topology.n_molecules)
    print("Box vectors (nm):\n", packed_topology.box_vectors.to(unit.nanometer))
    print("Positions shape:", packed_topology.get_positions().shape)

Protein radius: 3.114 nm
Dodecahedron scale: 8.229 nm
Box volume: 394.03 nm^3
Packing waters: 8933, counterions: 5
Total molecules packed: 8940
Box vectors (nm):
 [[8.228988377106717 0.0 0.0] [0.0 8.228988377106717 0.0] [4.114494188553358 4.114494188553358 5.818773483757442]] nanometer
Positions shape: (29743, 3)


In [15]:
if seq_type == "binder":
    # packed_topology.to_file(f"binders/seq{ID}_binder/packed_{box_shape}_seq{ID}_b.pdb")
    # nglview.show_structure_file(f"binders/seq{ID}_binder/packed_{box_shape}_seq{ID}_b.pdb")
    packed_topology.to_file(f"binders/pair_{ID}_binder/packed_{box_shape}_{ID}.pdb")
    nglview.show_structure_file(f"binders/pair_{ID}_binder/packed_{box_shape}_{ID}.pdb")
elif seq_type == "nonbinder":
    # packed_topology.to_file(f"nonbinders/seq{ID}_nb/packed_{box_shape}_seq{ID}_nb.pdb")
    # nglview.show_structure_file(f"nonbinders/seq{ID}_nb/packed_{box_shape}_seq{ID}_nb.pdb")
    packed_topology.to_file(f"nonbinders/pair_{ID}_nb/packed_{box_shape}_{ID}.pdb")
    nglview.show_structure_file(f"nonbinders/pair_{ID}_nb/packed_{box_shape}_{ID}.pdb")
elif seq_type == "neg_low_pkt":
    packed_topology.to_file(f"neg_low_pkt/pair_{ID}_low_pkt/packed_{box_shape}_pair{ID}.pdb")
    nglview.show_structure_file(f"neg_low_pkt/pair_{ID}_low_pkt/packed_{box_shape}_pair{ID}.pdb")
elif seq_type == "neg_fail_gate":
    packed_topology.to_file(f"neg_fail_gate/pair_{ID}_fail_gate/packed_{box_shape}_pair{ID}.pdb")
    nglview.show_structure_file(f"neg_fail_gate/pair_{ID}_fail_gate/packed_{box_shape}_pair{ID}.pdb")

In [16]:
topology_molecules = [water] * n_water
if counterion is not None:
    topology_molecules += [counterion] * n_counterions

In [17]:
# create Interchange for box of water
water_intrcg = Interchange.from_smirnoff(
    force_field=ForceField("openff_unconstrained-2.0.0.offxml"),
    topology=topology_molecules,
)

### Put everything together 
Dock protein+ligand complex with the solvent

In [18]:
system_intrcg = docked_intrcg.combine(water_intrcg)
system_intrcg.positions = packed_topology.get_positions()
system_intrcg.box = packed_topology.box_vectors

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/operations/_combine.py:183: InterchangeCombinationWarning: Setting positions to None because one or both objects added together were missing positions.
  warnings.warn(


In [19]:
w = system_intrcg.visualize()
w.clear_representations()
w.add_unitcell()
# Protein rep
w.add_representation(
    "licorice",
    radius=0.2,
    selection=[*range(protein_intrcg.topology.n_atoms)],
)
# Ligand rep
w.add_representation(
    "spacefill",
    selection=[*range(protein_intrcg.topology.n_atoms, docked_intrcg.topology.n_atoms)],
)
# Water rep
w.add_representation(
    "licorice",
    radius=0.1,
    selection=[*range(docked_intrcg.topology.n_atoms, system_intrcg.topology.n_atoms)],
)
w.center()
w

NGLWidget()

## Export to GROMACS

In [20]:
os.getcwd()
os.chdir(f'/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/{folder_name}/{name_prefix}_{ID}_{extension}')
os.getcwd()

'/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/nonbinders/pair_1654_nb'

In [21]:
# Hydrogen mass repartitioning
if seq_type == "binder":
    #system_intrcg.to_gromacs(prefix=f"binders/seq{ID}_binder/seq{ID}_b_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
    system_intrcg.to_gromacs(prefix=f"{name_prefix}_{ID}_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
elif seq_type == "nonbinder":
    #system_intrcg.to_gromacs(prefix=f"nonbinders/seq{ID}_nb/seq{ID}_nb_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
    system_intrcg.to_gromacs(prefix=f"{name_prefix}_{ID}_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
elif seq_type == "neg_low_pkt":
    system_intrcg.to_gromacs(prefix=f"pair{ID}_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)
elif seq_type == "neg_fail_gate":
    system_intrcg.to_gromacs(prefix=f"pair{ID}_{box_shape}_HMR", decimal=3, hydrogen_mass=3, monolithic=False)

/Users/ivanatang/miniforge3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/mdconfig.py:503: UserWarning: Ambiguous failure while processing constraints. Constraining h-bonds as a stopgap.
  warnings.warn(
